In [8]:
import pandas as pd

df = pd.read_csv("../week01/expenses.csv")
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")

## 5 — `.apply()` vs vectorized

Same result, two roads. One of them is much more expensive.
First prove they agree, then measure the gap.

In [9]:
slow = df["amount"].apply(lambda x: "yes" if x > 500 else "no")
fast = (df["amount"] > 500).map({True: "yes", False: "no"})

df["big"] = fast
slow.equals(fast)

True

In [10]:
big_df = pd.concat([df] * 20000, ignore_index=True)
len(big_df)

400000

In [11]:
%%timeit
big_df["amount"].apply(lambda x: "yes" if x > 500 else "no")

79.2 ms ± 844 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
(big_df["amount"] > 500).map({True: "yes", False: "no"})

6.22 ms ± 255 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Ratio — the interview answer

Fill in from the two `%%timeit` cells above, on 400,000 rows:

- `.apply()` — ___ ms
- vectorized — ___ ms
- **`.apply()` is ___x slower**

Why: `.apply()` runs the Python lambda once per row — 400,000 separate
interpreter calls, each boxing a value into a Python object. The vectorized
version does the comparison once in C over the whole array.

(Reference run on this machine: 69 ms vs 6.4 ms ≈ **11x**. Your numbers
will differ — write down your own.)

## 6 — logcat: top 5 noisiest tags

Same question as Week 1's `Counter.most_common(5)`, asked in pandas.
The two cells below must agree on the counts.

Note: where counts tie, `groupby` and `Counter` can order the tied tags
differently. Counts equal = correct. Order of a tie = not a bug.

In [17]:
import sys
from dataclasses import asdict

sys.path.insert(0,"../week01")
from logcat_parser import parse_file, noisiest_tags

lines = parse_file("../week01/logcat.txt")
logs = pd.DataFrame([asdict(l) for l in lines])

logs.groupby("tag").size().sort_values(ascending=False).head(3)

tag
SurfaceFlinger         11
GraphicsEnvironment     5
dc.asaanconnect         5
dtype: int64

In [7]:
noisiest_tags(lines)

[('SurfaceFlinger', 11),
 ('dc.asaanconnect', 5),
 ('GraphicsEnvironment', 5),
 ('TransportR...ortBackend', 4),
 ('WindowManager', 3)]